In [1]:
conn = psycopg2.connect("dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402")
video_path = 'Модуль А/fallback2.mp4'

In [1]:
import cv2
import os

# Принудительно устанавливаем переменную окружения для ffmpeg (иногда помогает в Windows)
os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "rtsp_transport;udp" 

# Пробуем разные варианты URL
# 1. С указанием порта и пути
# 2. Попробуйте добавить параметры таймаута
VIDEO_SOURCE = "rtmp://nntc.nnov.ru:5566/stream"

cap = cv2.VideoCapture(VIDEO_SOURCE, cv2.CAP_FFMPEG)

if not cap.isOpened():
    print("❌ Поток все еще недоступен. Проверьте: открыт ли порт 5566 в локальной сети колледжа?")
else:
    print("✅ Поток успешно захвачен!")

❌ Поток все еще недоступен. Проверьте: открыт ли порт 5566 в локальной сети колледжа?


In [ ]:
import requests # Не забудь добавить в начало файла

# Перед циклом while задаем таймер:
last_upload_time = time.time()
CLOUD_URL = "https://api.yourservice.com/upload" # Замени на URL из задания

while cap.isOpened():
    # ... тут идет чтение кадра и логика детекции (res = model.track...)
    
    now_ts = time.time()
    
    # === НОВЫЙ БЛОК: ОТПРАВКА В ОБЛАКО ===
    if now_ts - last_upload_time >= 30.0: # Прошло 30 секунд
        # Берем кадр с нарисованными рамками
        annotated_frame = res.plot() 
        
        # Кодируем кадр в формат JPEG в памяти (без сохранения на диск)
        _, buffer = cv2.imencode('.jpg', annotated_frame)
        
        try:
            # Отправляем POST запрос. Файл передается как multipart/form-data
            response = requests.post(
                CLOUD_URL, 
                files={"file": ("frame.jpg", buffer.tobytes(), "image/jpeg")},
                timeout=5 # Таймаут, чтобы видео не зависло, если облако тупит
            )
            print(f"Кадр отправлен в облако! Статус: {response.status_code}")
        except Exception as e:
            print(f"Ошибка отправки в облако: {e}")
            
        # Обновляем таймер
        last_upload_time = now_ts
    # ======================================
    # Облако S3
S3_ENDPOINT = "https://storage.yandexcloud.net"
S3_ACCESS_KEY = "ВАШ_КЛЮЧ"
S3_SECRET_KEY = "ВАШ_СЕКРЕТ"
S3_BUCKET_NAME = "traffic-frames-bucket"
# Скриншот в облако каждые 30 секунд
   if time.time() - last_upload_time >= 30:
        fname = f"frame_{now.strftime('%H%M%S')}.jpg"
        cv2.imwrite(fname, annotated_frame)
        try:
            s3_client.upload_file(fname, S3_BUCKET_NAME, f"records/{fname}")
            print(f"☁️ Кадр {fname} в S3")
        except: pass
        finally: 
            if os.path.exists(fname): os.remove(fname)
        last_upload_time = time.time()

In [14]:
points = []

def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x}, {y})")
        points.append((x, y))

cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()

cv2.imshow("Frame", frame)
cv2.setMouseCallback("Frame", click_event)

cv2.waitKey(0)
cv2.destroyAllWindows()

(0, 684)
(1798, 232)
(1759, 454)


In [44]:
import cv2
import psycopg2
import time
import numpy as np
import os
from ultralytics import YOLO
from datetime import datetime, timedelta
from minio import Minio
import io

# --- 1. КОНФИГУРАЦИЯ (НАСТРОЙКИ) ---
RTMP_URL = 'rtmp://nntc.nnov.ru:5566/stream'
VIDEO_PATH = 'Модуль А/fallback2.mp4'
DB_URL = "dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402"

# --- NEW: Настройка ROI в процентах (0.0 - 1.0) ---
# Точки трапеции по часовой стрелке. Настройте эти значения под свою полосу!
ROI_PERCENT_POINTS = [
    (0, 0.60),   # Левый дальний угол
    (0.75, 0.20),   # Правый дальний угол
    (1, 1),   # Правый ближний угол
    (0, 1)    # Левый ближний угол
]

# Параметры замера скорости и инцидентов
REAL_DIST_M = 15.0
LINE_THRESHOLD = 25  
PIXELS_PER_METER = 35
CRITICAL_DIST_M = 3.5
INCIDENT_COOLDOWN = 2
UPLOAD_INTERVAL_MIN = 30 

# Линии замера (в пикселях, так как они привязаны к конкретному ракурсу)
LINE_START_P1, LINE_START_P2 = (720, 430), (1236, 817)
LINE_END_P1, LINE_END_P2 = (1070, 321), (1606, 474)

# --- 2. ИНИЦИАЛИЗАЦИЯ ИСТОЧНИКОВ ---
MINIO_BUCKET = "user08"
MINIO_CLIENT = Minio("2.nntc.nnov.ru:9000", access_key="user08", secret_key="6f9tmxqp4zrb", secure=False)

print("🚀 Попытка подключения к потоку...")
cap = cv2.VideoCapture(RTMP_URL)
if not cap.isOpened():
    print("⚠️ Стрим недоступен. Запуск локального видео...")
    cap = cv2.VideoCapture(VIDEO_PATH)

model = YOLO('yolo12s.pt')

db_active = False
try:
    conn = psycopg2.connect(DB_URL, connect_timeout=2)
    conn.autocommit = True
    cursor = conn.cursor()
    db_active = True
    print("✅ БД: Подключено.")
except Exception:
    print("⚠️ БД: Автономный режим.")

# Служебные переменные
total_cars_count = 0
entry_times = {}
final_speeds = {}
processed_ids = set()
last_incident_time = {}
last_upload_time = datetime.now() - timedelta(minutes=UPLOAD_INTERVAL_MIN)

# --- 3. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---
def get_pixel_coords(percent_points, w, h):
    """Преобразует проценты в реальные пиксели кадра"""
    return np.array([(int(x * w), int(y * h)) for x, y in percent_points], np.int32)

def point_line_distance(px, py, x1, y1, x2, y2):
    return abs((y2 - y1)*px - (x2 - x1)*py + x2*y1 - y2*x1) / np.sqrt((y2 - y1)**2 + (x2 - x1)**2)

def log_event(message):
    print(message)
    with open("local_log.txt", "a", encoding="utf-8") as f:
        f.write(f"{datetime.now()} | {message}\n")

def upload_snapshot(frame):
    try:
        if not MINIO_CLIENT.bucket_exists(MINIO_BUCKET):
            MINIO_CLIENT.make_bucket(MINIO_BUCKET)
        _, buf = cv2.imencode('.jpg', frame)
        io_buf = io.BytesIO(buf)
        name = f"snap_{datetime.now().strftime('%H%M%S')}.jpg"
        MINIO_CLIENT.put_object(MINIO_BUCKET, name, io_buf, len(buf), content_type="image/jpeg")
        log_event(f"☁️ MinIO: Отправлен кадр {name}")
    except Exception as e: log_event(f"❌ MinIO Error: {e}")

# --- 4. ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ ---
try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        h, w = frame.shape[:2]
        now_ts, now_dt = time.time(), datetime.now()
        roi_pixels = get_pixel_coords(ROI_PERCENT_POINTS, w, h)
        display_frame = frame.copy()
        
        results = model.track(frame, persist=True, verbose=False, classes=[2, 3, 5, 7])

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(float)
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            active_objects = []

            for i in range(len(ids)):
                x1, y1, x2, y2 = boxes[i]
                t_id, cx, cy = int(ids[i]), (x1 + x2) / 2, (y1 + y2) / 2

                # ФИЛЬТР: Только если центр объекта внутри процентного ROI
                if cv2.pointPolygonTest(roi_pixels, (int(cx), int(cy)), False) < 0:
                    continue 

                active_objects.append({'id': t_id, 'cx': cx, 'cy': cy, 'box': (x1, y1, x2, y2)})

                # Отрисовка детекции
                cv2.rectangle(display_frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                
                # Логика скорости
                d_start = point_line_distance(cx, cy, *LINE_START_P1, *LINE_START_P2)
                d_end = point_line_distance(cx, cy, *LINE_END_P1, *LINE_END_P2)

                if d_start < LINE_THRESHOLD and t_id not in entry_times:
                    entry_times[t_id] = now_ts

                if d_end < LINE_THRESHOLD and t_id in entry_times and t_id not in processed_ids:
                    duration = now_ts - entry_times[t_id]
                    if duration > 0.3:
                        speed = (REAL_DIST_M / duration) * 3.6
                        processed_ids.add(t_id)
                        final_speeds[t_id] = speed
                        total_cars_count += 1
                        log_event(f"ID {t_id}: {speed:.1f} km/h")
                        if db_active:
                            try: cursor.execute("INSERT INTO full_tracking_data (video_id, track_id, speed_km_h) VALUES (12, %s, %s)", (t_id, float(speed)))
                            except: db_active = False

                label = f"ID: {t_id}" + (f" | {final_speeds[t_id]:.1f} km/h" if t_id in final_speeds else "")
                cv2.putText(display_frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Проверка опасных сближений (только для объектов в ROI)
            for i in range(len(active_objects)):
                for j in range(i + 1, len(active_objects)):
                    o1, o2 = active_objects[i], active_objects[j]
                    dist_m = (((o1['cx']-o2['cx'])**2 + (o1['cy']-o2['cy'])**2)**0.5) / PIXELS_PER_METER
                    if dist_m < CRITICAL_DIST_M:
                        pair = tuple(sorted((o1['id'], o2['id'])))
                        if pair not in last_incident_time or (now_ts - last_incident_time[pair]) > INCIDENT_COOLDOWN:
                            log_event(f"⚠️ DANGER: {o1['id']} & {o2['id']} ({dist_m:.2f}m)")
                            if db_active:
                                try: cursor.execute("INSERT INTO dangerous_incidents (track_id1, track_id2, distance) VALUES (%s, %s, %s)", (o1['id'], o2['id'], float(dist_m)))
                                except: db_active = False
                            last_incident_time[pair] = now_ts

        # --- 5. ВИЗУАЛИЗАЦИЯ И ИНТЕРФЕЙС ---
        # Отрисовка полупрозрачной зоны ROI
        overlay = display_frame.copy()
        cv2.fillPoly(overlay, [roi_pixels], (255, 191, 0))
        cv2.addWeighted(overlay, 0.25, display_frame, 0.75, 0, display_frame)
        cv2.polylines(display_frame, [roi_pixels], True, (255, 191, 0), 2)

        # Линии и счетчик
        cv2.line(display_frame, LINE_START_P1, LINE_START_P2, (0, 255, 0), 2)
        cv2.line(display_frame, LINE_END_P1, LINE_END_P2, (0, 0, 255), 2)
        cv2.rectangle(display_frame, (10, 10), (450, 75), (0, 0, 0), -1)
        cv2.putText(display_frame, f"TRAFFIC: {total_cars_count}", (25, 55), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)

        # Таймер облака
        if (now_dt - last_upload_time).total_seconds() >= UPLOAD_INTERVAL_MIN * 60:
            upload_snapshot(display_frame)
            last_upload_time = now_dt

        cv2.imshow('Traffic Analytics System', cv2.resize(display_frame, (1280, 720)))
        if cv2.waitKey(1) & 0xFF == ord('q'): break

finally:
    cap.release()
    cv2.destroyAllWindows()
    if db_active:
        cursor.close()
        conn.close()

🚀 Попытка подключения к потоку...
⚠️ Стрим недоступен. Запуск локального видео...
✅ БД: Подключено.
⚠️ DANGER: 3 & 7 (3.22m)
☁️ MinIO: Отправлен кадр snap_154730.jpg
⚠️ DANGER: 3 & 7 (3.19m)
⚠️ DANGER: 3 & 4 (3.39m)
⚠️ DANGER: 3 & 7 (3.20m)
⚠️ DANGER: 3 & 4 (2.50m)
⚠️ DANGER: 3 & 7 (3.21m)
⚠️ DANGER: 34 & 70 (2.78m)
⚠️ DANGER: 10 & 70 (2.59m)
⚠️ DANGER: 34 & 10 (3.13m)
⚠️ DANGER: 3 & 7 (3.28m)
⚠️ DANGER: 34 & 70 (3.32m)
⚠️ DANGER: 4 & 5 (3.38m)
⚠️ DANGER: 4 & 95 (2.86m)
⚠️ DANGER: 5 & 95 (0.01m)
ID 4: 8.5 km/h
⚠️ DANGER: 3 & 7 (3.27m)
⚠️ DANGER: 3 & 34 (3.35m)
⚠️ DANGER: 3 & 7 (3.31m)
⚠️ DANGER: 70 & 130 (2.76m)
⚠️ DANGER: 3 & 34 (3.19m)
⚠️ DANGER: 112 & 70 (3.36m)
⚠️ DANGER: 4 & 141 (0.01m)
⚠️ DANGER: 3 & 7 (3.21m)
⚠️ DANGER: 70 & 130 (2.87m)
ID 34: 10.5 km/h
⚠️ DANGER: 3 & 7 (3.22m)
⚠️ DANGER: 3 & 7 (3.24m)
⚠️ DANGER: 3 & 112 (3.47m)
⚠️ DANGER: 3 & 7 (3.30m)
⚠️ DANGER: 3 & 112 (3.37m)


KeyboardInterrupt: 